# Day 32 — **Dataclass: CrewAI Task & Agent Schema**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

CrewAI describes a crew declaratively: each **Agent** has a role, goal and backstory; each **Task** has a description, an expected output, tools, and an owning agent. Those are pure *data* — no behaviour, just fields with rules. That's exactly what `@dataclass` is for, and modelling it yourself is the fastest way to understand what CrewAI is doing before you install it.

Today:
1. `Task` as a `@dataclass` with `field(default_factory=...)`.
2. `__post_init__` **validation** — a task with no tools is a bug.
3. A crew run **serialised to JSON** for the audit trail.
4. **Dataclass vs Pydantic** — where each belongs in an agent stack.

## 1. `Task` as a dataclass

Four fields, mirroring CrewAI's real signature. Mutable defaults (`tools`) need `field(default_factory=list)` — a bare `[]` default would be shared by every instance.

In [1]:
from dataclasses import dataclass, field, asdict, fields

@dataclass
class Task:
    """A unit of work handed to one agent, mirroring CrewAI's Task."""
    description: str
    expected_output: str
    tools: list[str] = field(default_factory=list)
    agent: str | None = None

t = Task(
    description="Analyse Barclays Q4 2024 results and extract the key revenue drivers.",
    expected_output="5 bullet points, each with a figure and a source line.",
    tools=["pdf_reader", "web_search"],
    agent="researcher",
)
print(t, "\n")
print("fields:", [f.name for f in fields(Task)])
print("independent defaults:", Task("a", "b").tools is Task("c", "d").tools)

Task(description='Analyse Barclays Q4 2024 results and extract the key revenue drivers.', expected_output='5 bullet points, each with a figure and a source line.', tools=['pdf_reader', 'web_search'], agent='researcher') 

fields: ['description', 'expected_output', 'tools', 'agent']
independent defaults: False


☝ `field(default_factory=list)` gives each `Task` its own list — `False` above would mean two tasks silently sharing tools, a bug you'd chase for an hour. The free `__repr__` matters more than it looks: when a crew run goes wrong, printing the task is your first debugging step, and dataclasses make that readable by default.

## 2. `__post_init__` — validate at construction

A task whose `tools` list is empty can't do anything; an agent with no goal can't be routed to. Catch it where the object is *built*, not three LLM calls later.

In [2]:
@dataclass
class Task:
    description: str
    expected_output: str
    tools: list[str] = field(default_factory=list)
    agent: str | None = None

    def __post_init__(self) -> None:
        if not self.tools:
            raise ValueError(f"Task {self.description[:30]!r}: tools must not be empty")
        if len(self.description) < 15:
            raise ValueError("Task description too vague for an LLM to act on")

@dataclass(frozen=True, slots=True)
class Agent:
    """A crew member. Frozen — a role must not mutate mid-run."""
    role: str
    goal: str
    backstory: str = ""

    def __post_init__(self) -> None:
        if not self.goal:
            raise ValueError(f"Agent {self.role!r}: goal is required")

ok = Task("Summarise the Q4 revenue bridge in plain English.", "3 bullets", ["pdf_reader"], "analyst")
print("valid task ->", ok.agent, ok.tools)

for bad in [
    lambda: Task("Summarise the Q4 revenue bridge in plain English.", "3 bullets", []),
    lambda: Task("do stuff", "something", ["pdf_reader"]),
    lambda: Agent(role="writer", goal=""),
]:
    try:
        bad()
    except ValueError as exc:
        print("rejected ->", exc)

valid task -> analyst ['pdf_reader']
rejected -> Task 'Summarise the Q4 revenue bridg': tools must not be empty
rejected -> Task description too vague for an LLM to act on
rejected -> Agent 'writer': goal is required


☝ `__post_init__` runs after the generated `__init__` assigns the fields, so it's the natural home for cross-field rules. Note it fires on `frozen=True` classes too — construction-time validation still works when mutation doesn't. In a crew, this converts "the writer agent produced nothing and we don't know why" into an error at *definition* time, before a single token is spent.

## 3. Serialise the crew result — the audit trail

A bank needs to answer "which agent produced this, using which tools, at what time". `asdict()` walks nested dataclasses recursively; `json.dumps(default=...)` handles the types JSON doesn't know, like `datetime`.

In [3]:
import json
from datetime import datetime, timezone

@dataclass
class TaskResult:
    task: Task
    agent: Agent
    output: str
    finished_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

@dataclass
class CrewRun:
    crew: str
    results: list[TaskResult] = field(default_factory=list)

researcher = Agent("researcher", "Find the numbers behind the headline", "Ex-equity analyst.")
task = Task("Analyse Barclays Q4 2024 results and list revenue drivers.",
            "5 bullets with figures", ["pdf_reader"], "researcher")

run = CrewRun(crew="q4_research", results=[
    TaskResult(task, researcher, output="• Income up on NII\n• Costs flat\n• Impairments down")
])

audit = json.dumps(asdict(run), indent=2, default=str)
print(audit[:520], "...\n")
print("nested dataclasses flattened:", list(asdict(run)["results"][0].keys()))

{
  "crew": "q4_research",
  "results": [
    {
      "task": {
        "description": "Analyse Barclays Q4 2024 results and list revenue drivers.",
        "expected_output": "5 bullets with figures",
        "tools": [
          "pdf_reader"
        ],
        "agent": "researcher"
      },
      "agent": {
        "role": "researcher",
        "goal": "Find the numbers behind the headline",
        "backstory": "Ex-equity analyst."
      },
      "output": "\u2022 Income up on NII\n\u2022 Costs flat\n\u2022 Impa ...

nested dataclasses flattened: ['task', 'agent', 'output', 'finished_at']


☝ One `asdict()` on the top-level `CrewRun` flattens `Task` and `Agent` too — you get the whole run as plain dicts, ready for S3 or a database. `default=str` is what stops `datetime` raising `TypeError` mid-serialisation. This JSON blob *is* the audit trail: who ran, with what instructions, which tools, and when.

## 4. Dataclass or Pydantic? Both, in different places

Dataclasses do **not** coerce or check types at runtime — the annotation is a hint. Pydantic does. That difference decides where each belongs.

In [4]:
from pydantic import BaseModel, Field, ValidationError

sneaky = Task("Analyse the Q4 numbers carefully.", "3 bullets", ["pdf_reader"], agent=42)
print("dataclass accepted agent=42 ->", repr(sneaky.agent), type(sneaky.agent).__name__)

class TaskModel(BaseModel):
    description: str = Field(min_length=15)
    expected_output: str
    tools: list[str] = Field(min_length=1)
    agent: str | None = None

try:
    TaskModel(description="Analyse the Q4 numbers carefully.", expected_output="3 bullets",
              tools=["pdf_reader"], agent=42)
except ValidationError as exc:
    print("pydantic rejected ->", exc.errors()[0]["msg"])

parsed = TaskModel.model_validate_json(
    '{"description": "Analyse the Q4 numbers carefully.", "expected_output": "3 bullets", "tools": ["pdf_reader"]}'
)
print("parsed from untrusted JSON ->", parsed.tools, parsed.agent)

dataclass accepted agent=42 -> 42 int
pydantic rejected -> Input should be a valid string
parsed from untrusted JSON -> ['pdf_reader'] None


☝ The dataclass happily stored `agent=42` because annotations aren't enforced at runtime — fine for objects *you* construct in code, dangerous for anything crossing a trust boundary. The split that holds up in production: **Pydantic at the edges** (LLM output, API payloads, YAML crew configs — anywhere data arrives untrusted), **dataclasses inside** (internal, hot-path, zero dependency, cheaper to allocate). Real CrewAI uses Pydantic for exactly this reason: your crew definitions often come from YAML.

## 5. Recap — the crew is a schema before it's an LLM call

| Need | Tool | Why |
|---|---|---|
| Internal task/agent objects | `@dataclass` | free `__init__`/`__repr__`, cheap, stdlib |
| Own list/dict per instance | `field(default_factory=list)` | avoids shared mutable defaults |
| Cross-field rules | `__post_init__` | fails at construction, not mid-run |
| Immutable roles | `frozen=True, slots=True` | no mid-run mutation, lower memory |
| Audit trail | `asdict()` + `json.dumps(default=str)` | recursive, regulator-friendly |
| Untrusted input (YAML, LLM, API) | Pydantic `BaseModel` | actually enforces types |

Modelling `Agent` and `Task` yourself shows the real lesson: **a crew is a data structure**, and most crew bugs are schema bugs — an empty tool list, a vague description, a `None` agent. Validate at construction and the LLM layer gets much less interesting to debug. Today's module puts these schemas to work in an actual crew.